# 01 – Data collection
**Project Group 37 – Data Analytics HS2026 (ZHAW)**
*Marc Stump, Ayushan Ahilan*

**Research question:** Which factors influence whether a train arrives ≥ 3 minutes late – and do these factors differ between **German-speaking Switzerland and Romandy** and between **peak and off-peak hours**?

| # | Source | Access | Content |
|---|---|---|---|
| 1 | Actual data v2 (“Ist-Daten”), opentransportdata.swiss | CKAN web API → daily files (CSV) | scheduled and actual times of all public transport stops |
| 2 | Service points (atlas), opentransportdata.swiss | CKAN web API → CSV | coordinates, canton and municipality of every station |
| 3 | Open-Meteo Historical Weather API | REST API (JSON) | hourly weather per station |

The data is **queried, filtered and stored in SQLite by ourselves via web APIs** – no ready-made analysis dataset is used.

> Depending on the internet connection, the full collection takes approx. 30–90 minutes (14 × ~500 MB). Recommended: run `python run_collection.py` in the terminal. The cells below show the API calls step by step.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.db import get_connection

pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
config.OUTPUT_DIR.mkdir(exist_ok=True)
conn = get_connection()
print("Database:", config.DB_PATH.relative_to(ROOT))

Database: data/train_delays.sqlite


## 1. Actual data: query the available days via the CKAN API
The platform publishes one CSV file per operating day. The CKAN API method `package_show` returns the list of all files including their download URL.

In [2]:
from src import collect_actual_data
from src.http_utils import get_json

try:
    response = get_json(f"{config.CKAN_BASES[0]}/package_show", params={"id": config.ACTUAL_DATA_PACKAGE})
    resources = response["result"]["resources"]
    print("Number of resources:", len(resources))
    print("Example URL:", resources[-1].get("url"))
    available = collect_actual_data.list_available_days()
    print(f"{len(available)} days available: {min(available)} to {max(available)}")
    print("Selected for the analysis:", [str(d) for d in collect_actual_data.choose_days(available)])
except Exception as e:
    print("API not reachable:", e)

  Error (403 Client Error: Forbidden for url: https://data.opentransportdata.swiss/api/3/action/package_show?id=ist-daten-v2); retrying in 5s ...


  Error (403 Client Error: Forbidden for url: https://data.opentransportdata.swiss/api/3/action/package_show?id=ist-daten-v2); retrying in 10s ...


API not reachable: 403 Client Error: Forbidden for url: https://data.opentransportdata.swiss/api/3/action/package_show?id=ist-daten-v2


## 2. Weather: example query to Open-Meteo (Zürich HB)

In [3]:
try:
    example = get_json(config.OPEN_METEO_URL, params={
        "latitude": 47.3782, "longitude": 8.5402,
        "start_date": "2026-10-01", "end_date": "2026-10-01",
        "hourly": "temperature_2m,precipitation,snowfall,wind_speed_10m",
        "timezone": "Europe/Zurich"})
    display(pd.DataFrame(example["hourly"]).head(10))
except Exception as e:
    print("API not reachable:", e)

,time,temperature_2m,precipitation,snowfall,wind_speed_10m
0,2026-10-01T00:00,17.5,0.0,0.0,4.1
1,2026-10-01T01:00,17.5,0.0,0.0,2.5
2,2026-10-01T02:00,17.4,0.0,0.0,4.5
3,2026-10-01T03:00,17.4,0.0,0.0,1.7
4,2026-10-01T04:00,17.1,0.0,0.0,3.1
5,2026-10-01T05:00,16.6,0.0,0.0,2.5
6,2026-10-01T06:00,16.6,0.0,0.0,3.6
7,2026-10-01T07:00,16.2,0.0,0.0,1.4
8,2026-10-01T08:00,18.4,0.0,0.0,3.8
9,2026-10-01T09:00,20.3,0.0,0.0,0.2


## 3. Full data collection
Set `RUN_COLLECTION = True` to load everything directly from the notebook (alternatively run `python run_collection.py` in the terminal). Days that are already loaded are skipped.

Steps:
1. **Stations** – load the service point CSV, keep Swiss stations with coordinates, assign the language region
2. **Actual data** – download each day, filter in chunks (trains only, no pass-throughs, Swiss stops only), compute delay = actual − scheduled, store in table `stops`, delete the raw file
3. **Weather** – query hourly weather data for the 100 busiest stations

In [4]:
RUN_COLLECTION = False

if RUN_COLLECTION:
    from src import collect_stations, collect_weather
    collect_stations.collect(conn)
    collect_actual_data.collect(conn, n_days=config.N_DAYS)
    collect_weather.collect(conn, n=config.TOP_N_STATIONS)

## 4. Check of the collected data (SQL queries from Python)
Our collection covers 14–27 October 2026. No daily file was published for 19 October 2026, so the database contains 13 operating days.

In [5]:
display(pd.read_sql("SELECT * FROM load_log ORDER BY source, key", conn))
display(pd.read_sql("""
    SELECT 'stations' AS table_name, COUNT(*) AS n_rows FROM stations
    UNION ALL SELECT 'stops', COUNT(*) FROM stops
    UNION ALL SELECT 'weather', COUNT(*) FROM weather""", conn))

,source,key,rows_raw,rows_saved,loaded_at
0,actual_data,2026-10-14,2601463,160767,2026-11-03T15:30:42
1,actual_data,2026-10-15,2598189,160046,2026-11-03T15:31:08
2,actual_data,2026-10-16,2660582,162656,2026-11-03T15:31:34
3,actual_data,2026-10-17,2168822,160094,2026-11-03T15:40:11
4,actual_data,2026-10-18,1697673,150358,2026-11-03T15:40:28
5,actual_data,2026-10-20,2581567,161714,2026-11-03T15:40:58
6,actual_data,2026-10-21,2583004,162098,2026-11-03T15:41:25
7,actual_data,2026-10-22,2570215,161928,2026-11-03T15:41:59
8,actual_data,2026-10-23,2622716,164924,2026-11-03T15:42:27
9,actual_data,2026-10-24,2144641,157194,2026-11-03T15:42:49


,table_name,n_rows
0,stations,31095
1,stops,2078429
2,weather,36000


In [6]:
display(pd.read_sql("SELECT * FROM stops LIMIT 5", conn))
display(pd.read_sql("SELECT * FROM stations LIMIT 5", conn))
display(pd.read_sql("SELECT * FROM weather LIMIT 5", conn))

,operating_day,trip_id,operator,line,train_type,bpuic,stop_name,arr_scheduled,arr_actual,arr_status,dep_scheduled,dep_actual,dep_status,arr_delay_min,dep_delay_min,cancelled,extra_trip
0,2026-10-14,80:800693:3040:000,DB Regio,RE3,RE,8503424,Schaffhausen,2026-10-14 05:54:00,2026-10-14 05:55:17,REAL,2026-10-14 05:56:00,2026-10-14 05:56:19,REAL,1.28,0.32,0,0
1,2026-10-14,80:800693:3041:000,DB Regio,RE3,RE,8503424,Schaffhausen,2026-10-14 06:14:00,2026-10-14 06:14:17,REAL,2026-10-14 06:16:00,2026-10-14 06:16:08,REAL,0.28,0.13,0,0
2,2026-10-14,80:800693:3042:000,DB Regio,RE3,RE,8503424,Schaffhausen,2026-10-14 06:58:00,2026-10-14 06:58:36,REAL,2026-10-14 07:00:00,2026-10-14 07:00:24,REAL,0.60,0.40,0,0
3,2026-10-14,80:800693:3043:000,DB Regio,RE3,RE,8503424,Schaffhausen,2026-10-14 06:58:00,2026-10-14 07:00:00,REAL,2026-10-14 07:00:00,2026-10-14 07:00:30,REAL,2.00,0.50,0,0
4,2026-10-14,80:800693:3044:000,DB Regio,RE3,RE,8503424,Schaffhausen,2026-10-14 07:58:00,2026-10-14 07:59:57,REAL,2026-10-14 08:00:00,2026-10-14 08:00:40,REAL,1.95,0.67,0,0


,bpuic,name,lat,lon,canton,municipality,region
0,8500001,Conche (Ayent),46.320006,7.399290,VS,Ayent,Romandy
1,8500002,Moay (télésiège),46.058118,7.198132,VS,Val de Bagnes,Romandy
2,8500003,Chaux Ronde (Gryon),46.293616,7.115991,VD,Gryon,Romandy
3,8500004,"Selva, via Alpsu spv.",46.662107,8.723795,GR,Tujetsch,German-speaking
4,8500005,"Camischolas, via Alpsu/Strem",46.679657,8.764866,GR,Tujetsch,German-speaking


,bpuic,time,temperature,precipitation,snowfall,wind
0,8503000,2026-10-14 00:00,12.9,0.1,0.0,4.0
1,8503000,2026-10-14 01:00,12.0,0.1,0.0,2.1
2,8503000,2026-10-14 02:00,12.4,0.0,0.0,3.2
3,8503000,2026-10-14 03:00,11.6,0.0,0.0,2.6
4,8503000,2026-10-14 04:00,11.6,0.1,0.0,0.6
